# M2: GraphSAGE의 CLV 수준·구성·가격 좌표 임베딩 (Dunnhumby, seed 42)

직전 LightGCN·NGCF·GAT 이식과 동일한 `ID(64) + CLV 관계(2) + 가격(1)` layer-0 입력을 2층 mean-aggregating GraphSAGE에 이식합니다. `GraphSAGE@64`, 동일 총차원의 `GraphSAGE@67`, 실제 CLV M2, degree-matched CLV 순열을 고정 100 epoch로 비교합니다. 최종 test와 holdout은 구성하지 않습니다.

네 arm은 순차 실행되며 epoch checkpoint에서 자동 재개됩니다. 판정 기준은 GraphSAGE@67 대비 정확도 유지와 가격·구매금액 가중 적중값 개선, 그리고 실제 CLV가 degree-matched shuffle을 고CLV Recall/NDCG에서 모두 이기는지입니다. 실패하면 같은 개발구간에서 하이퍼파라미터를 조정하지 않고 M2 백본 이식을 종료합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
SOURCE_COMMIT = 'd411b8ff31d652ed47a88ac2b7c5e3786cfd5da9'
REPO_DIR = '/content/clv-m2-lightgcn-runner'
if len(SOURCE_COMMIT) != 40:
    raise RuntimeError('검토된 소스 커밋을 SOURCE_COMMIT에 고정해야 합니다')
!if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch origin; else git clone {REPO_URL} {REPO_DIR}; fi
!git -C {REPO_DIR} checkout {SOURCE_COMMIT}
%cd {REPO_DIR}
!git rev-parse HEAD

# 같은 런타임에서 재실행해도 checkout한 수정본을 다시 읽습니다.
import sys
for module_name in list(sys.modules):
    if module_name.startswith('graphsage_clv_'):
        del sys.modules[module_name]

In [ ]:
import json
import torch
from graphsage_clv_level_composition_price_screen import (
    configure_graphsage_clv_screen,
    preflight_summary,
    run_graphsage_clv_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_graphsage_clv_screen()
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
result_df = run_graphsage_clv_screen(cfg)

In [ ]:
import pandas as pd
from IPython.display import display

print('1) 절대지표: LightGCN 참고, GraphSAGE@64, GraphSAGE@67, 실제 CLV, degree-matched shuffle')
display(result_df)
print('2) GraphSAGE 내부 대조군별 성과 비교')
display(pd.DataFrame(result_df.attrs['comparison']))
print('3) GraphSAGE@67 및 shuffle 대비 실제 CLV Top-10 변경')
display(pd.DataFrame(result_df.attrs['top10_overlap']))
print('4) 사전 판정 규칙 결과')
print(json.dumps(result_df.attrs['screening_reading'], ensure_ascii=False, indent=2))
print('5) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))